# 基于不同存储结构的求和算子实现实验

顺序存储和链式存储是数据结构课程中最基础、也最适合与高性能计算联系起来的两个存储视角。本实验围绕长度为 `N` 的数值序列求和展开，在 Ascend C 环境中分别实现顺序存储版本和链式存储版本的求和算子，并用性能测试观察它们在 NPU 侧的执行差异。

本实验学习大纲如下：

1. 环境准备：创建实验目录并加载 CANN 环境；
2. 问题分析：说明两种存储结构的数据布局、访问方式和实验参数；
3. 核函数开发：实现顺序存储求和 kernel 和链式存储求和 kernel；
4. 正确性验证：准备 Host 输入和 CPU 参考结果，开发核函数调用代码，完成构建、运行、误差校验和性能分析；
5. 实验总结：归纳数据布局、访存局部性和计算性能之间的关系。


---
## 1. 环境准备

首先创建实验所需目录，并尝试加载 Ascend CANN 环境变量。

目录划分如下：

- `Source/01.01/Code/include`：保存 Host 侧公共头文件。
- `Source/01.01/Code`：保存输入生成、CPU 参考计算、计时和结果统计代码。
- `Source/01.01/ascend_ops/op_kernel`：保存 Device 侧核函数代码。
- `Source/01.01/ascend_ops/host_launch`：保存 Host 侧核函数调用代码。
- `Source/01.01/scripts`：保存构建、运行和 profiling 脚本。
- `Source/01.01/results`：保存实验结果。

如果当前环境已安装 CANN，下面的代码会加载 `set_env.sh`。如果没有找到该文件，仍然可以继续生成代码，但完整编译和运行需要在已配置 CANN 的环境中完成。


In [ ]:
!mkdir -p Source/01.01/Code/include
!mkdir -p Source/01.01/scripts
!mkdir -p Source/01.01/results
!mkdir -p Source/01.01/ascend_ops/host_launch
!mkdir -p Source/01.01/ascend_ops/op_kernel

import os
import subprocess
from pathlib import Path

candidate_paths = [
    os.environ.get("ASCEND_TOOLKIT_HOME"),
    os.environ.get("ASCEND_INSTALL_PATH"),
    "/usr/local/Ascend/ascend-toolkit/latest",
]

set_env = None
for item in candidate_paths:
    if item and Path(item, "set_env.sh").exists():
        set_env = Path(item, "set_env.sh")
        break

if set_env is not None:
    env = subprocess.check_output(
        f"bash -l -c 'source {set_env} && env'",
        shell=True,
        text=True,
    )
    for line in env.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    print("Ascend environment loaded from:", set_env)
else:
    print("Ascend set_env.sh was not found. Code generation can continue; build/run needs CANN.")

print("Experiment directory:", Path("Source/01.01").resolve())


---
## 2. 问题分析

### 2.1 输入、输出和数据布局

本实验实现序列求和：

$$
S=\sum_{i=0}^{N-1}x_i
$$

逻辑上，两种实现处理的是同一条长度为 `N` 的线性表；差别仅在于物理存储布局不同。

1. 顺序存储  
输入为连续 `float` 数组，索引为 `values[i]`。kernel 只需从头到尾线性遍历并累加。

2. 链式存储  
CPU 侧先构造静态链表。每个结点包含：

- `value`：当前结点数值；
- `next`：后继结点索引。

为了突出链式存储的访存不连续特征，默认会把逻辑相邻结点打乱到不同物理位置，再用 `next` 串成一条完整链。NPU 侧则分别接收结点值数组和后继索引数组，并按 `head -> next -> next` 的方式跳转访问。


### 2.2 实验参数与性能观察目标

本实验的 Host 参数由 `SumConfig` 管理，主要包括：

- `n`：序列长度；
- `warmup/repeat`：预热与正式计时次数；
- `seed`：随机输入种子；
- `shuffle_nodes`：是否打乱链表结点的物理顺序。

实验的核心观察目标不是“谁的算术运算更多”，而是“相同求和任务在不同数据布局上的访存代价差异”：

- 顺序存储版本具有连续访问特征，通常会更快；
- 链式存储版本需要频繁跟随 `next` 跳转，通常会更慢；
- 若关闭打乱，使链表结点在物理上接近连续，则两者差距可能缩小。


### 2.3 Host 公共参数：`storage_sum_config.hpp`

`SumConfig` 保存输入规模、计时参数和随机种子；`SumMetrics` 统一记录每条路径的平均时间、最小时间、读取数据量估算和误差结果。


In [ ]:
%%writefile Source/01.01/Code/include/storage_sum_config.hpp

#pragma once

#include <cstddef>
#include <cstdint>
#include <string>

namespace storagesum {

enum class StorageKind { Sequence, Linked };

struct SumConfig {
    std::size_t n = 65536;
    int warmup = 2;
    int repeat = 5;
    unsigned seed = 20260701u;
    bool shuffle_nodes = true;
    bool verbose = false;
    std::string csv_path;
};

struct SumMetrics {
    std::string kernel;
    std::string storage;
    std::size_t n = 0;
    int warmup = 0;
    int repeat = 0;
    double avg_us = 0.0;
    double min_us = 0.0;
    double estimated_read_mb = 0.0;
    double got = 0.0;
    double ref = 0.0;
    double abs_err = 0.0;
    bool pass = false;
};

std::string to_string(StorageKind kind);
StorageKind parse_storage_kind(const std::string& text);

}  // namespace storagesum


---
## 3. 核函数开发

本实验的 Device 侧只包含两个 kernel 入口：

1. `sequence_sum`：顺序存储求和；
2. `linked_sum`：链式存储求和。

这两个 kernel 都固定使用 1 个 AI Core 启动，从而尽量控制变量，把差异聚焦到数据访问模式本身，而不是并行策略。


### 3.1 顺序存储求和 kernel

`sequence_sum` 直接绑定一段连续 `GlobalTensor<float>`，然后执行线性遍历：

- `valuesGm.GetValue(i)` 读取第 `i` 个元素；
- 使用一个 `float acc` 做累加；
- 最后把结果写回长度为 1 的输出缓冲区。

这是最接近“连续数组扫描”的求和路径。


In [ ]:
%%writefile Source/01.01/ascend_ops/op_kernel/sequence_sum.cpp
#include "kernel_operator.h"

using namespace AscendC;

extern "C" __global__ __aicore__ void sequence_sum(
    GM_ADDR values,
    GM_ADDR out,
    uint32_t n)
{
    InitSocState();

    if (GetBlockIdx() != 0) {
        return;
    }

    GlobalTensor<float> valuesGm;
    GlobalTensor<float> outGm;
    valuesGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(values), n);
    outGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(out), 1);

    float acc = 0.0f;
    for (uint32_t i = 0; i < n; ++i) {
        acc += valuesGm.GetValue(i);
    }

    outGm.SetValue(0, acc);
    DataCacheCleanAndInvalid<float, CacheLine::ENTIRE_DATA_CACHE,
                             DcciDst::CACHELINE_OUT>(outGm);
}


### 3.2 链式存储求和 kernel

`linked_sum` 接收三类输入：

- `nodeValues`：结点值数组；
- `nodeNext`：后继索引数组；
- `head`：链表头结点位置。

kernel 通过 `current = nextGm.GetValue(idx)` 不断跳转，直到到达链尾或访问次数达到 `n`。  
这里的 `visited < n` 和 `current >= n` 既是边界保护，也能避免异常链表结构造成死循环。


In [ ]:
%%writefile Source/01.01/ascend_ops/op_kernel/linked_sum.cpp
#include "kernel_operator.h"

using namespace AscendC;

extern "C" __global__ __aicore__ void linked_sum(
    GM_ADDR nodeValues,
    GM_ADDR nodeNext,
    GM_ADDR out,
    uint32_t n,
    int32_t head)
{
    InitSocState();

    if (GetBlockIdx() != 0) {
        return;
    }

    GlobalTensor<float> valuesGm;
    GlobalTensor<int32_t> nextGm;
    GlobalTensor<float> outGm;
    valuesGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(nodeValues), n);
    nextGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(nodeNext), n);
    outGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(out), 1);

    float acc = 0.0f;
    int32_t current = head;
    uint32_t visited = 0;
    while (current >= 0 &&
           static_cast<uint32_t>(current) < n &&
           visited < n) {
        const uint64_t idx = static_cast<uint64_t>(current);
        acc += valuesGm.GetValue(idx);
        current = nextGm.GetValue(idx);
        ++visited;
    }

    outGm.SetValue(0, acc);
    DataCacheCleanAndInvalid<float, CacheLine::ENTIRE_DATA_CACHE,
                             DcciDst::CACHELINE_OUT>(outGm);
}


---
## 4. 核函数运行验证

Host 负责生成输入、计算 CPU 参考结果、初始化 ACL、申请 Device 内存、分别启动两条路径并验证结果。

与样例矩阵乘法不同，这里没有多核和 Tiling 版本；实验重点是“同样的串行求和，不同的存储布局会导致怎样的性能差异”。


### 4.1 Host 输入与 CPU 参考结果

`StorageInput` 同时保存：

- `logical_values`：逻辑上的原始序列；
- `sequence_values`：顺序存储数组；
- `linked_nodes`：链表结点数组；
- `linked_head`：链表头结点索引。

CPU 参考结果统一直接对 `logical_values` 求和，从而确保两条 Device 路径面对的是同一批逻辑数据。


In [ ]:
%%writefile Source/01.01/Code/include/data_utils.hpp

#pragma once

#include "storage_sum_config.hpp"

#include <cstdint>
#include <vector>

namespace storagesum {

struct LinkedNodeHost {
    float value = 0.0f;
    int32_t next = -1;
};

struct StorageInput {
    std::vector<float> logical_values;
    std::vector<float> sequence_values;
    std::vector<LinkedNodeHost> linked_nodes;
    int32_t linked_head = -1;
};

StorageInput make_input(const SumConfig& cfg);
double cpu_reference_sum(const std::vector<float>& values);
double cpu_sequence_sum(const StorageInput& input);
double cpu_linked_sum(const StorageInput& input);

}  // namespace storagesum


#### 4.1.2 计时工具：`timer.hpp`

CPU 演示程序和公共性能统计都使用同一个简单的微秒级计时器。


In [ ]:
%%writefile Source/01.01/Code/include/timer.hpp

#pragma once

#include <chrono>

namespace storagesum {

class Timer {
public:
    void tic();
    double toc_us() const;

private:
    std::chrono::high_resolution_clock::time_point start_{};
};

}  // namespace storagesum


#### 4.1.3 Host 公共统计接口：`sum_core.hpp`

`sum_core.hpp` 对外暴露顺序和链式两种 CPU 求和路径的运行接口，以及表格和 CSV 输出接口。它主要用于本地 CPU 演示程序。


In [ ]:
%%writefile Source/01.01/Code/include/sum_core.hpp

#pragma once

#include "data_utils.hpp"

#include <string>

namespace storagesum {

SumMetrics run_sequence_sum(const StorageInput& input, const SumConfig& cfg);
SumMetrics run_linked_sum(const StorageInput& input, const SumConfig& cfg);
void write_csv(const std::string& path, const SumMetrics& seq, const SumMetrics& linked);
void print_metrics_table(const SumMetrics& seq, const SumMetrics& linked);

}  // namespace storagesum


#### 4.1.4 参数字符串解析：`storage_sum_config.cpp`

这里把 `sequence/linked` 枚举转换和字符串解析集中实现，便于后续扩展更多存储方式。


In [ ]:
%%writefile Source/01.01/Code/storage_sum_config.cpp

#include "storage_sum_config.hpp"

#include <algorithm>
#include <cctype>
#include <stdexcept>

namespace storagesum {

namespace {

std::string lower_copy(std::string value) {
    std::transform(value.begin(), value.end(), value.begin(), [](unsigned char c) {
        return static_cast<char>(std::tolower(c));
    });
    return value;
}

}  // namespace

std::string to_string(StorageKind kind) {
    return kind == StorageKind::Sequence ? "sequence" : "linked";
}

StorageKind parse_storage_kind(const std::string& text) {
    const std::string v = lower_copy(text);
    if (v == "sequence" || v == "seq" || v == "array") return StorageKind::Sequence;
    if (v == "linked" || v == "list" || v == "chain" || v == "static-linked") return StorageKind::Linked;
    throw std::invalid_argument("unsupported storage kind: " + text);
}

}  // namespace storagesum


#### 4.1.5 计时实现：`timer.cpp`

计时器实现非常简单，只负责记录起始时间并返回微秒级耗时。


In [ ]:
%%writefile Source/01.01/Code/timer.cpp

#include "timer.hpp"

namespace storagesum {

void Timer::tic() {
    start_ = std::chrono::high_resolution_clock::now();
}

double Timer::toc_us() const {
    const auto end = std::chrono::high_resolution_clock::now();
    return std::chrono::duration<double, std::micro>(end - start_).count();
}

}  // namespace storagesum


#### 4.1.6 构造顺序与链式存储结构：`data_utils.cpp`

`make_input` 是本实验最关键的 Host 输入准备函数：

1. 先生成逻辑顺序一致的 `logical_values`；
2. 顺序存储路径直接复制为连续数组；
3. 链式路径先生成物理位置排列 `physical_order`；
4. 若启用 `shuffle_nodes`，则打乱物理位置；
5. 再根据逻辑顺序把 `next` 串起来，得到一条逻辑链表。

这样做的意义是：逻辑序列不变，但链表结点在物理上可以是不连续的。


In [ ]:
%%writefile Source/01.01/Code/data_utils.cpp

#include "data_utils.hpp"

#include <algorithm>
#include <numeric>
#include <random>
#include <stdexcept>

namespace storagesum {

StorageInput make_input(const SumConfig& cfg) {
    if (cfg.n == 0) {
        throw std::invalid_argument("n must be positive");
    }

    StorageInput input;
    input.logical_values.resize(cfg.n);
    input.sequence_values.resize(cfg.n);

    std::mt19937 rng(cfg.seed);
    std::uniform_real_distribution<float> dist(-1.0f, 1.0f);

    for (std::size_t i = 0; i < cfg.n; ++i) {
        const float v = dist(rng);
        input.logical_values[i] = v;
        input.sequence_values[i] = v;
    }

    std::vector<std::size_t> physical_order(cfg.n);
    std::iota(physical_order.begin(), physical_order.end(), 0);
    if (cfg.shuffle_nodes && cfg.n > 1) {
        std::shuffle(physical_order.begin(), physical_order.end(), rng);
    }

    input.linked_nodes.resize(cfg.n);
    std::vector<std::size_t> logical_to_physical(cfg.n, 0);
    for (std::size_t logical_idx = 0; logical_idx < cfg.n; ++logical_idx) {
        const std::size_t physical_idx = physical_order[logical_idx];
        logical_to_physical[logical_idx] = physical_idx;
        input.linked_nodes[physical_idx].value = input.logical_values[logical_idx];
    }

    for (std::size_t logical_idx = 0; logical_idx < cfg.n; ++logical_idx) {
        const std::size_t physical_idx = logical_to_physical[logical_idx];
        const int32_t next = (logical_idx + 1 < cfg.n)
            ? static_cast<int32_t>(logical_to_physical[logical_idx + 1])
            : -1;
        input.linked_nodes[physical_idx].next = next;
    }
    input.linked_head = static_cast<int32_t>(logical_to_physical[0]);
    return input;
}

double cpu_reference_sum(const std::vector<float>& values) {
    float sum = 0.0f;
    for (float v : values) {
        sum += v;
    }
    return static_cast<double>(sum);
}

double cpu_sequence_sum(const StorageInput& input) {
    float sum = 0.0f;
    for (float v : input.sequence_values) {
        sum += v;
    }
    return static_cast<double>(sum);
}

double cpu_linked_sum(const StorageInput& input) {
    float sum = 0.0f;
    int32_t current = input.linked_head;
    std::size_t visited = 0;
    while (current >= 0) {
        if (static_cast<std::size_t>(current) >= input.linked_nodes.size()) {
            throw std::out_of_range("linked node next index out of range");
        }
        const auto& node = input.linked_nodes[static_cast<std::size_t>(current)];
        sum += node.value;
        current = node.next;
        ++visited;
        if (visited > input.linked_nodes.size()) {
            throw std::runtime_error("linked list contains a cycle");
        }
    }
    return static_cast<double>(sum);
}

}  // namespace storagesum


#### 4.1.7 CPU 侧指标统计：`sum_core.cpp`

`sum_core.cpp` 主要完成三件事：

1. 统一 warmup/repeat 流程；
2. 估算顺序存储与链式存储的读取数据量；
3. 输出可直接比较的性能表格和 CSV。

这里的 `estimated_read_mb` 不是硬件真实带宽统计，而是按数据结构大小给出的近似访问量，用于帮助学生理解链式结构为什么至少要额外读取 `next` 域。


In [ ]:
%%writefile Source/01.01/Code/sum_core.cpp

#include "sum_core.hpp"

#include "timer.hpp"

#include <algorithm>
#include <cmath>
#include <filesystem>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <numeric>
#include <stdexcept>
#include <vector>

namespace storagesum {

namespace {

double estimate_sequence_read_mb(std::size_t n) {
    return static_cast<double>(n * sizeof(float)) / (1024.0 * 1024.0);
}

double estimate_linked_read_mb(std::size_t n) {
    return static_cast<double>(n * (sizeof(float) + sizeof(int32_t))) / (1024.0 * 1024.0);
}

template <typename Fn>
SumMetrics run_sum_case(const char* kernel_name,
                        StorageKind kind,
                        std::size_t n,
                        int warmup,
                        int repeat,
                        double ref,
                        double estimated_read_mb,
                        Fn&& fn) {
    if (repeat <= 0 || warmup < 0) {
        throw std::invalid_argument("warmup must be >= 0 and repeat must be > 0");
    }

    double got = 0.0;
    for (int i = 0; i < warmup; ++i) {
        got = fn();
    }

    Timer timer;
    std::vector<double> times;
    times.reserve(static_cast<std::size_t>(repeat));
    for (int i = 0; i < repeat; ++i) {
        timer.tic();
        got = fn();
        times.push_back(timer.toc_us());
    }

    const double avg = std::accumulate(times.begin(), times.end(), 0.0) /
                       static_cast<double>(times.size());
    const double minv = *std::min_element(times.begin(), times.end());
    const double abs_err = std::abs(got - ref);

    SumMetrics row;
    row.kernel = kernel_name;
    row.storage = to_string(kind);
    row.n = n;
    row.warmup = warmup;
    row.repeat = repeat;
    row.avg_us = avg;
    row.min_us = minv;
    row.estimated_read_mb = estimated_read_mb;
    row.got = got;
    row.ref = ref;
    row.abs_err = abs_err;
    row.pass = abs_err <= 1e-5 * std::max(1.0, std::abs(ref));
    return row;
}

}  // namespace

SumMetrics run_sequence_sum(const StorageInput& input, const SumConfig& cfg) {
    const double ref = cpu_reference_sum(input.logical_values);
    return run_sum_case("cpu_sequence_sum",
                        StorageKind::Sequence,
                        cfg.n,
                        cfg.warmup,
                        cfg.repeat,
                        ref,
                        estimate_sequence_read_mb(cfg.n),
                        [&]() { return cpu_sequence_sum(input); });
}

SumMetrics run_linked_sum(const StorageInput& input, const SumConfig& cfg) {
    const double ref = cpu_reference_sum(input.logical_values);
    return run_sum_case("cpu_linked_sum",
                        StorageKind::Linked,
                        cfg.n,
                        cfg.warmup,
                        cfg.repeat,
                        ref,
                        estimate_linked_read_mb(cfg.n),
                        [&]() { return cpu_linked_sum(input); });
}

void write_csv(const std::string& path, const SumMetrics& seq, const SumMetrics& linked) {
    const std::filesystem::path out_path(path);
    if (!out_path.parent_path().empty()) {
        std::filesystem::create_directories(out_path.parent_path());
    }
    std::ofstream out(path);
    if (!out) {
        throw std::runtime_error("failed to open csv: " + path);
    }
    out << "kernel,storage,n,warmup,repeat,avg_us,min_us,estimated_read_mb,got,ref,abs_err,status\n";
    const SumMetrics rows[2] = {seq, linked};
    for (const auto& r : rows) {
        out << r.kernel << ',' << r.storage << ',' << r.n << ','
            << r.warmup << ',' << r.repeat << ','
            << std::fixed << std::setprecision(6)
            << r.avg_us << ',' << r.min_us << ',' << r.estimated_read_mb << ','
            << r.got << ',' << r.ref << ',' << r.abs_err << ','
            << (r.pass ? "PASS" : "FAIL") << '\n';
    }
}

void print_metrics_table(const SumMetrics& seq, const SumMetrics& linked) {
    std::cout << std::left
              << std::setw(22) << "kernel"
              << std::setw(12) << "storage"
              << std::setw(10) << "N"
              << std::right
              << std::setw(14) << "avg_us"
              << std::setw(14) << "min_us"
              << std::setw(16) << "read_MB"
              << std::setw(16) << "abs_err"
              << std::setw(10) << "status" << '\n';
    const SumMetrics rows[2] = {seq, linked};
    for (const auto& r : rows) {
        std::cout << std::left
                  << std::setw(22) << r.kernel
                  << std::setw(12) << r.storage
                  << std::setw(10) << r.n
                  << std::right
                  << std::fixed << std::setprecision(3)
                  << std::setw(14) << r.avg_us
                  << std::setw(14) << r.min_us
                  << std::setw(16) << r.estimated_read_mb
                  << std::scientific << std::setprecision(3)
                  << std::setw(16) << r.abs_err
                  << std::fixed
                  << std::setw(10) << (r.pass ? "PASS" : "FAIL") << '\n';
    }
}

}  // namespace storagesum


### 4.2 CPU 演示程序：`main.cpp`

`main.cpp` 主要用于本地 CPU 演示。它负责：

1. 解析 `--n/--warmup/--repeat/--seed` 等参数；
2. 构造两种输入数据；
3. 分别运行顺序存储和链式存储的 CPU 参考路径；
4. 输出两行对比结果。

这个程序本身不是 Ascend NPU 程序，但很适合在课程中先讲清楚数据结构和性能差异的基本逻辑。


In [ ]:
%%writefile Source/01.01/Code/main.cpp

#include "data_utils.hpp"
#include "sum_core.hpp"

#include <cstdlib>
#include <exception>
#include <iostream>
#include <stdexcept>
#include <string>

namespace storagesum {
namespace {

void print_help(const char* argv0) {
    std::cout
        << "Usage: " << argv0 << " [options]\n\n"
        << "Options:\n"
        << "  --n <int>              sequence length, default 65536\n"
        << "  --warmup <int>         warmup iterations, default 2\n"
        << "  --repeat <int>         measured iterations, default 5\n"
        << "  --seed <int>           random seed, default 20260701\n"
        << "  --ordered-linked       linked nodes keep physical order; default is shuffled\n"
        << "  --csv <path>           write the two results as csv\n"
        << "  --verbose              print additional notes\n"
        << "  --help                 show this message\n";
}

std::size_t parse_size(const std::string& s, const std::string& key) {
    const auto value = std::stoull(s);
    if (value == 0) throw std::invalid_argument(key + " must be positive");
    return static_cast<std::size_t>(value);
}

}  // namespace

SumConfig parse_args(int argc, char** argv) {
    SumConfig cfg;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto need_value = [&](const std::string& key) -> std::string {
            if (i + 1 >= argc) throw std::invalid_argument("missing value after " + key);
            return argv[++i];
        };
        if (arg == "--help" || arg == "-h") {
            print_help(argv[0]);
            std::exit(0);
        } else if (arg == "--n") {
            cfg.n = parse_size(need_value(arg), arg);
        } else if (arg == "--warmup") {
            cfg.warmup = std::stoi(need_value(arg));
        } else if (arg == "--repeat") {
            cfg.repeat = std::stoi(need_value(arg));
        } else if (arg == "--seed") {
            cfg.seed = static_cast<unsigned>(std::stoul(need_value(arg)));
        } else if (arg == "--ordered-linked") {
            cfg.shuffle_nodes = false;
        } else if (arg == "--csv") {
            cfg.csv_path = need_value(arg);
        } else if (arg == "--verbose") {
            cfg.verbose = true;
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    if (cfg.warmup < 0 || cfg.repeat <= 0) {
        throw std::invalid_argument("warmup must be >= 0 and repeat must be > 0");
    }
    return cfg;
}
}  // namespace storagesum

int main(int argc, char** argv) {
    try {
        auto cfg = storagesum::parse_args(argc, argv);
        if (cfg.verbose) {
            std::cout << "[info] generating sequence data and static linked-list data...\n";
        }
        const auto input = storagesum::make_input(cfg);
        const auto seq = storagesum::run_sequence_sum(input, cfg);
        const auto linked = storagesum::run_linked_sum(input, cfg);
        storagesum::print_metrics_table(seq, linked);
        if (!cfg.csv_path.empty()) {
            storagesum::write_csv(cfg.csv_path, seq, linked);
            std::cout << "[info] csv written to " << cfg.csv_path << '\n';
        }
        return (seq.pass && linked.pass) ? 0 : 2;
    } catch (const std::exception& e) {
        std::cerr << "[error] " << e.what() << '\n';
        return 1;
    }
}


### 4.3 Host 侧核函数调用

NPU Host 程序需要完成以下工作：

1. 解析运行参数；
2. 构造顺序存储和链式存储数据；
3. 将顺序数组、链表值数组和链表后继数组拷贝到 Device；
4. 分别调用 `sequence_sum` 和 `linked_sum`；
5. 统计时间、校验结果并输出 CSV。

#### 4.3.1 头文件、ACL 错误检查与参数结构

`CaseResult` 用于统一保存每条 NPU 路径的指标。所有 ACL 返回值都由 `ACL_CHECK` 包装检查。


In [ ]:
%%writefile Source/01.01/ascend_ops/host_launch/storage_sum_npu_main.cpp
#include "data_utils.hpp"

#include <acl/acl.h>
#include "aclrtlaunch_linked_sum.h"
#include "aclrtlaunch_sequence_sum.h"

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdlib>
#include <filesystem>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <stdexcept>
#include <string>
#include <vector>

#define ACL_CHECK(expr)                                                                            \
    do {                                                                                           \
        aclError _ret = (expr);                                                                    \
        if (_ret != ACL_SUCCESS) {                                                                 \
            throw std::runtime_error(std::string("[ACL ERROR] ") + #expr +                       \
                                     " failed, ret=" + std::to_string(_ret) +                    \
                                     " at " + __FILE__ + ":" + std::to_string(__LINE__));       \
        }                                                                                          \
    } while (0)

namespace {

struct Options {
    int device = 0;
    uint32_t n = 65536;
    int warmup = 2;
    int repeat = 5;
    unsigned seed = 20260701;
    bool shuffleNodes = true;
    std::string csv = "results/storage_sum_result.csv";
};

struct CaseResult {
    std::string kernel;
    std::string storage;
    uint32_t n = 0;
    double avgUs = 0.0;
    double minUs = 0.0;
    double estimatedReadMb = 0.0;
    double got = 0.0;
    double ref = 0.0;
    double absErr = 0.0;
    bool pass = false;
};

uint32_t parse_u32(const char *s, const char *name)
{
    unsigned long v = std::stoul(s);
    if (v == 0 || v > static_cast<unsigned long>(std::numeric_limits<uint32_t>::max())) {
        throw std::invalid_argument(std::string(name) + " must be positive and fit in uint32_t");
    }
    return static_cast<uint32_t>(v);
}


#### 4.3.2 参数解析与访问量估算

本实验的 NPU 程序参数比较简单：

- `--n`：序列长度；
- `--warmup/--repeat`：预热与正式计时次数；
- `--seed`：随机种子；
- `--ordered-linked`：如果指定，则不打乱链表结点物理顺序。

访问量估算规则也很直观：

- 顺序存储：每个元素读取一个 `float`；
- 链式存储：每个结点至少读取一个 `float` 和一个 `int32_t next`。


In [ ]:
%%writefile -a Source/01.01/ascend_ops/host_launch/storage_sum_npu_main.cpp
Options parse_args(int argc, char **argv)
{
    Options opt;
    for (int i = 1; i < argc; ++i) {
        std::string arg = argv[i];
        auto need = [&](const char *name) -> const char * {
            if (i + 1 >= argc) throw std::invalid_argument(std::string("missing value after ") + name);
            return argv[++i];
        };
        if (arg == "--device") opt.device = std::stoi(need("--device"));
        else if (arg == "--n") opt.n = parse_u32(need("--n"), "n");
        else if (arg == "--warmup") opt.warmup = std::stoi(need("--warmup"));
        else if (arg == "--repeat") opt.repeat = std::stoi(need("--repeat"));
        else if (arg == "--seed") opt.seed = static_cast<unsigned>(std::stoul(need("--seed")));
        else if (arg == "--ordered-linked") opt.shuffleNodes = false;
        else if (arg == "--csv") opt.csv = need("--csv");
        else if (arg == "--help" || arg == "-h") {
            std::cout << "Usage: storage_sum_ascend_demo [--device 0] [--n 65536] "
                         "[--warmup 2] [--repeat 5] [--ordered-linked]\n";
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    if (opt.warmup < 0 || opt.repeat <= 0) {
        throw std::invalid_argument("warmup must be >= 0 and repeat must be > 0");
    }
    return opt;
}

double estimate_sequence_read_mb(uint32_t n)
{
    return static_cast<double>(n * sizeof(float)) / (1024.0 * 1024.0);
}

double estimate_linked_read_mb(uint32_t n)
{
    return static_cast<double>(n * (sizeof(float) + sizeof(int32_t))) / (1024.0 * 1024.0);
}

void check_result(double got, double ref, double &absErr, bool &pass)
{
    absErr = std::abs(got - ref);
    pass = absErr <= 1e-4 * std::max(1.0, std::abs(ref));
}


#### 4.3.3 启动顺序存储 kernel 与链式存储 kernel

`run_sequence` 和 `run_linked` 负责：

1. 在每次启动前把输出缓冲区置零；
2. 启动 kernel 并同步流；
3. 多次测量后统计平均时间和最小时间；
4. 将 Device 结果拷回 Host，与 CPU 参考结果对比。

两条路径唯一的本质差别就是输入数据的物理布局与 kernel 的访问方式。


In [ ]:
%%writefile -a Source/01.01/ascend_ops/host_launch/storage_sum_npu_main.cpp
CaseResult run_sequence(const Options &opt,
                        float *sequenceDevice,
                        float *outDevice,
                        aclrtStream stream,
                        double ref)
{
    auto run_once = [&]() -> double {
        const float zero = 0.0f;
        ACL_CHECK(aclrtMemcpy(outDevice, sizeof(float), &zero, sizeof(float), ACL_MEMCPY_HOST_TO_DEVICE));
        const auto t0 = std::chrono::high_resolution_clock::now();
        ACLRT_LAUNCH_KERNEL(sequence_sum)(1, stream, sequenceDevice, outDevice, opt.n);
        ACL_CHECK(aclrtSynchronizeStream(stream));
        const auto t1 = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(t1 - t0).count();
    };

    for (int i = 0; i < opt.warmup; ++i) (void)run_once();
    std::vector<double> times;
    times.reserve(static_cast<std::size_t>(opt.repeat));
    for (int i = 0; i < opt.repeat; ++i) times.push_back(run_once());

    float out = 0.0f;
    ACL_CHECK(aclrtMemcpy(&out, sizeof(float), outDevice, sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST));

    CaseResult r;
    r.kernel = "sequence_sum";
    r.storage = "sequence";
    r.n = opt.n;
    r.avgUs = std::accumulate(times.begin(), times.end(), 0.0) / static_cast<double>(times.size());
    r.minUs = *std::min_element(times.begin(), times.end());
    r.estimatedReadMb = estimate_sequence_read_mb(opt.n);
    r.got = static_cast<double>(out);
    r.ref = ref;
    check_result(r.got, r.ref, r.absErr, r.pass);
    return r;
}

CaseResult run_linked(const Options &opt,
                      float *linkedValuesDevice,
                      int32_t *linkedNextDevice,
                      float *outDevice,
                      int32_t head,
                      aclrtStream stream,
                      double ref)
{
    auto run_once = [&]() -> double {
        const float zero = 0.0f;
        ACL_CHECK(aclrtMemcpy(outDevice, sizeof(float), &zero, sizeof(float), ACL_MEMCPY_HOST_TO_DEVICE));
        const auto t0 = std::chrono::high_resolution_clock::now();
        ACLRT_LAUNCH_KERNEL(linked_sum)(1, stream, linkedValuesDevice, linkedNextDevice, outDevice, opt.n, head);
        ACL_CHECK(aclrtSynchronizeStream(stream));
        const auto t1 = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(t1 - t0).count();
    };

    for (int i = 0; i < opt.warmup; ++i) (void)run_once();
    std::vector<double> times;
    times.reserve(static_cast<std::size_t>(opt.repeat));
    for (int i = 0; i < opt.repeat; ++i) times.push_back(run_once());

    float out = 0.0f;
    ACL_CHECK(aclrtMemcpy(&out, sizeof(float), outDevice, sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST));

    CaseResult r;
    r.kernel = "linked_sum";
    r.storage = "linked";
    r.n = opt.n;
    r.avgUs = std::accumulate(times.begin(), times.end(), 0.0) / static_cast<double>(times.size());
    r.minUs = *std::min_element(times.begin(), times.end());
    r.estimatedReadMb = estimate_linked_read_mb(opt.n);
    r.got = static_cast<double>(out);
    r.ref = ref;
    check_result(r.got, r.ref, r.absErr, r.pass);
    return r;
}


#### 4.3.4 输出表格、写 CSV 和 Host 主函数

主函数流程如下：

1. 生成逻辑一致的两种输入；
2. 额外把 `linked_nodes` 拆成值数组和后继数组；
3. 初始化 ACL 与 Device；
4. 申请顺序数组、链表值数组、链表后继数组和输出缓冲区；
5. 依次运行顺序存储和链式存储两个 kernel；
6. 输出两行性能结果并释放资源。


In [ ]:
%%writefile -a Source/01.01/ascend_ops/host_launch/storage_sum_npu_main.cpp
void print_table(const std::vector<CaseResult> &rows)
{
    std::cout << std::left
              << std::setw(18) << "kernel"
              << std::setw(12) << "storage"
              << std::setw(10) << "N"
              << std::right
              << std::setw(14) << "avg_us"
              << std::setw(14) << "min_us"
              << std::setw(16) << "read_MB"
              << std::setw(16) << "abs_err"
              << std::setw(10) << "status" << '\n';
    for (const auto &r : rows) {
        std::cout << std::left
                  << std::setw(18) << r.kernel
                  << std::setw(12) << r.storage
                  << std::setw(10) << r.n
                  << std::right
                  << std::fixed << std::setprecision(3)
                  << std::setw(14) << r.avgUs
                  << std::setw(14) << r.minUs
                  << std::setw(16) << r.estimatedReadMb
                  << std::scientific << std::setprecision(3)
                  << std::setw(16) << r.absErr
                  << std::fixed
                  << std::setw(10) << (r.pass ? "PASS" : "FAIL") << '\n';
    }
}

void write_csv(const std::string &path, const std::vector<CaseResult> &rows)
{
    const std::filesystem::path outPath(path);
    if (!outPath.parent_path().empty()) {
        std::filesystem::create_directories(outPath.parent_path());
    }
    std::ofstream out(path);
    if (!out) {
        throw std::runtime_error("failed to open csv: " + path);
    }
    out << "kernel,storage,n,avg_us,min_us,estimated_read_mb,got,ref,abs_err,status\n";
    for (const auto &r : rows) {
        out << r.kernel << ',' << r.storage << ',' << r.n << ','
            << std::fixed << std::setprecision(6)
            << r.avgUs << ',' << r.minUs << ',' << r.estimatedReadMb << ','
            << r.got << ',' << r.ref << ',' << r.absErr << ','
            << (r.pass ? "PASS" : "FAIL") << '\n';
    }
}

} // namespace

int main(int argc, char **argv)
{
    bool aclInited = false;
    int activeDevice = -1;
    aclrtStream stream = nullptr;
    float *sequenceDevice = nullptr;
    float *linkedValuesDevice = nullptr;
    int32_t *linkedNextDevice = nullptr;
    float *outDevice = nullptr;

    auto cleanup = [&]() {
        if (sequenceDevice != nullptr) {
            (void)aclrtFree(sequenceDevice);
            sequenceDevice = nullptr;
        }
        if (linkedValuesDevice != nullptr) {
            (void)aclrtFree(linkedValuesDevice);
            linkedValuesDevice = nullptr;
        }
        if (linkedNextDevice != nullptr) {
            (void)aclrtFree(linkedNextDevice);
            linkedNextDevice = nullptr;
        }
        if (outDevice != nullptr) {
            (void)aclrtFree(outDevice);
            outDevice = nullptr;
        }
        if (stream != nullptr) {
            (void)aclrtDestroyStream(stream);
            stream = nullptr;
        }
        if (activeDevice >= 0) {
            (void)aclrtResetDevice(activeDevice);
            activeDevice = -1;
        }
        if (aclInited) {
            (void)aclFinalize();
            aclInited = false;
        }
    };

    try {
        const Options opt = parse_args(argc, argv);

        storagesum::SumConfig hostCfg;
        hostCfg.n = opt.n;
        hostCfg.warmup = opt.warmup;
        hostCfg.repeat = opt.repeat;
        hostCfg.seed = opt.seed;
        hostCfg.shuffle_nodes = opt.shuffleNodes;
        const auto input = storagesum::make_input(hostCfg);
        const double ref = storagesum::cpu_reference_sum(input.logical_values);

        std::vector<float> linkedValues(input.linked_nodes.size());
        std::vector<int32_t> linkedNext(input.linked_nodes.size());
        for (std::size_t i = 0; i < input.linked_nodes.size(); ++i) {
            linkedValues[i] = input.linked_nodes[i].value;
            linkedNext[i] = input.linked_nodes[i].next;
        }

        const size_t sequenceBytes = static_cast<size_t>(opt.n) * sizeof(float);
        const size_t linkedValueBytes = static_cast<size_t>(opt.n) * sizeof(float);
        const size_t linkedNextBytes = static_cast<size_t>(opt.n) * sizeof(int32_t);

        ACL_CHECK(aclInit(nullptr));
        aclInited = true;
        ACL_CHECK(aclrtSetDevice(opt.device));
        activeDevice = opt.device;
        ACL_CHECK(aclrtCreateStream(&stream));

        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&sequenceDevice), sequenceBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&linkedValuesDevice), linkedValueBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&linkedNextDevice), linkedNextBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&outDevice), sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST));

        ACL_CHECK(aclrtMemcpy(sequenceDevice, sequenceBytes,
                              input.sequence_values.data(), sequenceBytes,
                              ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemcpy(linkedValuesDevice, linkedValueBytes,
                              linkedValues.data(), linkedValueBytes,
                              ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemcpy(linkedNextDevice, linkedNextBytes,
                              linkedNext.data(), linkedNextBytes,
                              ACL_MEMCPY_HOST_TO_DEVICE));

        std::vector<CaseResult> rows;
        rows.push_back(run_sequence(opt, sequenceDevice, outDevice, stream, ref));
        rows.push_back(run_linked(opt, linkedValuesDevice, linkedNextDevice, outDevice,
                                  input.linked_head, stream, ref));

        print_table(rows);
        write_csv(opt.csv, rows);
        std::cout << "[info] csv written to " << opt.csv << '\n';

        const bool allPass = std::all_of(rows.begin(), rows.end(), [](const CaseResult &r) { return r.pass; });
        cleanup();
        return allPass ? 0 : 2;
    } catch (const std::exception &e) {
        cleanup();
        std::cerr << "[error] " << e.what() << '\n';
        return 1;
    }
}


### 4.4 CMake 构建配置

与样例项目一样，这里同时保留 CPU 演示路径和 Ascend C 实机路径：

- `storage_sum_demo`：本地 CPU 演示程序；
- `storage_sum_ascend_demo`：Ascend C Host + kernel 实验程序。

两个 Device kernel 被拆成独立的 Ascend C 编译目标，分别对应顺序存储求和和链式存储求和。


In [ ]:
%%writefile Source/01.01/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
project(ascendc_storage_sum_operator LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build Ascend C real-device experiment" OFF)

add_library(storage_sum_common
    Code/storage_sum_config.cpp
    Code/data_utils.cpp
    Code/sum_core.cpp
    Code/timer.cpp
)
target_include_directories(storage_sum_common PUBLIC Code/include)
target_compile_options(storage_sum_common PRIVATE
    $<$<CXX_COMPILER_ID:GNU,Clang>:-O3 -Wall -Wextra -Wpedantic>
)

add_executable(storage_sum_demo
    Code/main.cpp
)
target_link_libraries(storage_sum_demo PRIVATE storage_sum_common)
target_compile_options(storage_sum_demo PRIVATE
    $<$<CXX_COMPILER_ID:GNU,Clang>:-O3 -Wall -Wextra -Wpedantic>
)

if(BUILD_ASCEND)
  set(RUN_MODE "npu" CACHE STRING "Ascend C run mode: npu/cpu/sim")
  set(SOC_VERSION "ascend910b1" CACHE STRING "Ascend SOC version, e.g. ascend910b1/ascend910b2/ascend310p3")
  set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
  if(NOT ASCEND_CANN_PATH)
    set(ASCEND_CANN_PATH "/usr/local/Ascend/ascend-toolkit/latest" CACHE PATH "CANN installation path" FORCE)
  endif()
  set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
  set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

  if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  else()
    message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}. Check ASCEND_CANN_PATH/ASCEND_INSTALL_PATH.")
  endif()

  message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
  message(STATUS "SOC_VERSION=${SOC_VERSION}")
  include("${ASCENDC_CMAKE_FILE}")

  ascendc_library(storage_sum_sequence_kernels STATIC
      ascend_ops/op_kernel/sequence_sum.cpp
  )
  ascendc_include_directories(storage_sum_sequence_kernels PRIVATE
      ${CMAKE_CURRENT_SOURCE_DIR}/Code/include
  )
  ascendc_compile_definitions(storage_sum_sequence_kernels PRIVATE
      -DASCENDC_DUMP=0
  )

  ascendc_library(storage_sum_linked_kernels STATIC
      ascend_ops/op_kernel/linked_sum.cpp
  )
  ascendc_include_directories(storage_sum_linked_kernels PRIVATE
      ${CMAKE_CURRENT_SOURCE_DIR}/Code/include
  )
  ascendc_compile_definitions(storage_sum_linked_kernels PRIVATE
      -DASCENDC_DUMP=0
  )

  add_executable(storage_sum_ascend_demo
      ascend_ops/host_launch/storage_sum_npu_main.cpp
  )
  target_include_directories(storage_sum_ascend_demo PRIVATE
      Code/include
      ${ASCEND_CANN_PACKAGE_PATH}/include
      ${ASCEND_CANN_PACKAGE_PATH}/include/external
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
      ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp
      ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/tikcfw
      ${CMAKE_INSTALL_PREFIX}/include/storage_sum_sequence_kernels
      ${CMAKE_INSTALL_PREFIX}/include/storage_sum_linked_kernels
      ${CMAKE_BINARY_DIR}/out/include/storage_sum_sequence_kernels
      ${CMAKE_BINARY_DIR}/out/include/storage_sum_linked_kernels
  )
  target_link_directories(storage_sum_ascend_demo PRIVATE
      ${ASCEND_CANN_PACKAGE_PATH}/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/acllib/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/devlib
      ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/devlib
  )
  target_compile_options(storage_sum_ascend_demo PRIVATE
      $<$<CXX_COMPILER_ID:GNU,Clang>:-O3 -Wall -Wextra -Wno-deprecated-declarations>
  )
  target_compile_definitions(storage_sum_ascend_demo PRIVATE
      SOC_VERSION="${SOC_VERSION}"
  )
  target_link_libraries(storage_sum_ascend_demo PRIVATE
      storage_sum_sequence_kernels
      storage_sum_linked_kernels
      storage_sum_common
      ascendcl
      platform
      ascendalog
      c_sec
      dl
  )
  add_dependencies(storage_sum_ascend_demo
      storage_sum_sequence_kernels
      storage_sum_linked_kernels
  )
endif()

install(DIRECTORY ascend_ops DESTINATION share/ascendc_storage_sum_operator)


### 4.5 完整构建、运行与 profiling 脚本

`run.sh` 负责加载 CANN 环境、配置 CMake、编译并运行实验。  
`profile.sh` 负责在 `msprof` 环境下执行同一程序，生成 profiling 数据。


In [ ]:
%%writefile Source/01.01/scripts/run.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build_ascend"
RESULT_DIR="${SCRIPT_DIR}/results"
ASCEND_INSTALL_PATH_DEFAULT="/usr/local/Ascend/ascend-toolkit/latest"
ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH:-${ASCEND_INSTALL_PATH_DEFAULT}}"
SOC_VERSION="${SOC_VERSION:-ascend910b1}"
DEVICE_ID=0
RUN_MODE="npu"
BUILD_TYPE="Release"
CLEAN=1
WARMUP=2
REPEAT=5
N=65536
SEED=20260701
ORDERED_LINKED=0
EXTRA_ARGS=()

usage() {
  cat <<USAGE
Usage: bash scripts/run.sh [options] [-- extra_args_for_binary]

Options:
  -a <path>   ASCEND_INSTALL_PATH, default: /usr/local/Ascend/ascend-toolkit/latest
  -v <soc>    SOC_VERSION, default: ascend910b1
  -d <id>     device id, default: 0
  -n <num>    sequence length N, default: 65536
  -s <num>    random seed, default: 20260701
  -w <num>    warmup count, default: 2
  -r <num>    repeat count, default: 5
  -R <mode>   CMake run mode, default: npu
  -t <type>   CMake build type, default: Release
  -o          keep linked-list nodes in physical order instead of shuffled order
  -c          clean build directory before building; enabled by default
  -h          show help
USAGE
}

while getopts ":a:v:d:n:s:w:r:R:t:och" opt; do
  case ${opt} in
    a) ASCEND_INSTALL_PATH="${OPTARG}" ;;
    v) SOC_VERSION="${OPTARG}" ;;
    d) DEVICE_ID="${OPTARG}" ;;
    n) N="${OPTARG}" ;;
    s) SEED="${OPTARG}" ;;
    w) WARMUP="${OPTARG}" ;;
    r) REPEAT="${OPTARG}" ;;
    R) RUN_MODE="${OPTARG}" ;;
    t) BUILD_TYPE="${OPTARG}" ;;
    o) ORDERED_LINKED=1 ;;
    c) CLEAN=1 ;;
    h) usage; exit 0 ;;
    \?) echo "Unknown option: -${OPTARG}" >&2; usage; exit 1 ;;
    :) echo "Option -${OPTARG} requires a value." >&2; usage; exit 1 ;;
  esac
done
shift $((OPTIND - 1))

if [[ $# -gt 0 && "$1" == "--" ]]; then
  shift
fi
EXTRA_ARGS=("$@")

if [[ ! -d "${ASCEND_INSTALL_PATH}" ]]; then
  echo "ASCEND_INSTALL_PATH does not exist: ${ASCEND_INSTALL_PATH}" >&2
  exit 1
fi

if [[ -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]]; then
  source "${ASCEND_INSTALL_PATH}/set_env.sh"
fi

export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
export SOC_VERSION

if [[ "${CLEAN}" == "1" ]]; then
  rm -rf "${BUILD_DIR}"
fi
mkdir -p "${BUILD_DIR}" "${RESULT_DIR}"
cd "${BUILD_DIR}"

cmake "${SCRIPT_DIR}" \
  -DCMAKE_BUILD_TYPE="${BUILD_TYPE}" \
  -DBUILD_ASCEND=ON \
  -DASCEND_CANN_PATH="${ASCEND_INSTALL_PATH}" \
  -DASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}" \
  -DSOC_VERSION="${SOC_VERSION}" \
  -DRUN_MODE="${RUN_MODE}"

cmake --build . -j

BIN="${BUILD_DIR}/storage_sum_ascend_demo"
CSV="${RESULT_DIR}/storage_sum_result.csv"
CMD=("${BIN}" --device "${DEVICE_ID}" --n "${N}" \
     --warmup "${WARMUP}" --repeat "${REPEAT}" \
     --seed "${SEED}" --csv "${CSV}")
if [[ "${ORDERED_LINKED}" == "1" ]]; then
  CMD+=(--ordered-linked)
fi
CMD+=("${EXTRA_ARGS[@]}")

echo "[RUN] ${CMD[*]}"
"${CMD[@]}"


In [ ]:
%%writefile Source/01.01/scripts/profile.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build_ascend"
PROFILE_DIR="${SCRIPT_DIR}/results/profile"
ASCEND_INSTALL_PATH_DEFAULT="/usr/local/Ascend/ascend-toolkit/latest"
ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH:-${ASCEND_INSTALL_PATH_DEFAULT}}"
DEVICE_ID=0
N=65536
WARMUP=2
REPEAT=10
SEED=20260701
ORDERED_LINKED=0

usage() {
  cat <<USAGE
Usage: bash scripts/profile.sh [options]

Options:
  -a <path>   ASCEND_INSTALL_PATH, default: /usr/local/Ascend/ascend-toolkit/latest
  -d <id>     device id, default: 0
  -n <num>    sequence length N, default: 65536
  -s <num>    random seed, default: 20260701
  -w <num>    warmup count, default: 2
  -r <num>    repeat count, default: 10
  -o          keep linked-list nodes in physical order instead of shuffled order
  -h          show help
USAGE
}

while getopts ":a:d:n:s:w:r:oh" opt; do
  case ${opt} in
    a) ASCEND_INSTALL_PATH="${OPTARG}" ;;
    d) DEVICE_ID="${OPTARG}" ;;
    n) N="${OPTARG}" ;;
    s) SEED="${OPTARG}" ;;
    w) WARMUP="${OPTARG}" ;;
    r) REPEAT="${OPTARG}" ;;
    o) ORDERED_LINKED=1 ;;
    h) usage; exit 0 ;;
    \?) echo "Unknown option: -${OPTARG}" >&2; usage; exit 1 ;;
    :) echo "Option -${OPTARG} requires a value." >&2; usage; exit 1 ;;
  esac
done

if [[ -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]]; then
  source "${ASCEND_INSTALL_PATH}/set_env.sh"
fi

if ! command -v msprof >/dev/null 2>&1; then
  echo "msprof not found in PATH." >&2
  exit 1
fi

BIN="${BUILD_DIR}/storage_sum_ascend_demo"
if [[ ! -x "${BIN}" ]]; then
  echo "Cannot find executable: ${BIN}" >&2
  exit 1
fi

mkdir -p "${PROFILE_DIR}"

APP=("${BIN}" --device "${DEVICE_ID}" --n "${N}" --warmup "${WARMUP}" --repeat "${REPEAT}" --seed "${SEED}" --csv "${SCRIPT_DIR}/results/profile_run.csv")
if [[ "${ORDERED_LINKED}" == "1" ]]; then
  APP+=(--ordered-linked)
fi

echo "[PROFILE] msprof --output=${PROFILE_DIR} ${APP[*]}"
msprof --output="${PROFILE_DIR}" "${APP[@]}"


In [ ]:
!chmod +x Source/01.01/scripts/run.sh
!chmod +x Source/01.01/scripts/profile.sh
!find Source/01.01 -maxdepth 3 -type f | sort


### 4.6 运行完整实验

默认命令运行两条路径：

1. `sequence_sum`：顺序存储求和；
2. `linked_sum`：链式存储求和。

如果不加 `-o`，链表结点默认会被打乱到不同物理位置；这通常能更明显地观察到性能差异。


In [ ]:
!cd Source/01.01 && \
bash scripts/run.sh -n 65536 -w 2 -r 5


### 4.7 读取结果并计算性能倍率

结果保存在 `Source/01.01/results/storage_sum_result.csv`。下面读取两条路径的结果，并计算：

- 顺序存储相对于链式存储的加速比；
- 如果后续做 `--ordered-linked` 对比，也可以把两次结果一起比较。


In [ ]:
import csv
from pathlib import Path

csv_path = Path("Source/01.01/results/storage_sum_result.csv")
if not csv_path.exists():
    print("Result CSV does not exist yet:", csv_path)
    print("Please run the Ascend experiment cell first.")
else:
    with csv_path.open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        print(row)

    by_kernel = {row["kernel"]: row for row in rows}
    seq = by_kernel.get("sequence_sum")
    linked = by_kernel.get("linked_sum")
    if seq and linked:
        seq_us = float(seq["avg_us"])
        linked_us = float(linked["avg_us"])
        print(f"Speedup of sequence over linked: {linked_us / seq_us:.2f}x")


### 4.8 扩展实验

可以进一步设计如下对比：

1. 改变 `N`，观察数据规模增大时两种存储方式差距是否扩大；
2. 增加 `-o`，观察链表结点物理上接近连续时性能是否改善；
3. 继续扩展到树结构、二叉链表结构或其他基于指针/索引跳转的存储方式。


In [ ]:
experiments = [
    "bash scripts/run.sh -n 4096 -w 2 -r 10",
    "bash scripts/run.sh -n 65536 -w 2 -r 10",
    "bash scripts/run.sh -n 262144 -w 2 -r 10",
    "bash scripts/run.sh -n 65536 -w 2 -r 10 -o",
    "bash scripts/profile.sh -n 65536 -w 2 -r 20",
]

for cmd in experiments:
    print("cd Source/01.01 &&", cmd)


---
## 5. 实验总结

本实验实现了两条 Ascend NPU 路径：顺序存储求和和链式存储求和。

- 顺序存储版本展示了连续数组扫描的访问模式；
- 链式存储版本展示了基于 `next` 跳转访问的访存模式；
- 两条路径共享同一批逻辑输入和同一份 CPU 参考结果；
- 性能差异主要反映的是数据布局与访存局部性差异，而不是算术运算差异。

如果实验结果显示顺序存储显著快于链式存储，那么就很好地验证了本实验的教学目标：**存储结构不仅决定抽象数据组织方式，也会直接影响底层计算性能。**
